In [1]:
import pandas as pd
import sqlite3

con = sqlite3.connect('../data/checking-logs.sqlite')

In [2]:
# con.execute("""
# CREATE TABLE datamart AS
# SELECT  
#     c.uid,
#     c.labname,
#     c.timestamp AS first_commit_ts,
#     pv.first_view_ts
# FROM checker c
# LEFT JOIN (
#     SELECT uid, MIN(datetime) AS first_view_ts
#     FROM pageviews
#     WHERE uid LIKE 'user_%'
#     GROUP BY uid
# ) pv
# ON c.uid = pv.uid
# WHERE c.status = 'ready'
#   AND c.numTrials = 1
#   AND c.uid LIKE 'user_%'
#   AND c.labname IN (
#       'laba04',
#       'laba04s',
#       'laba05',
#       'laba06',
#       'laba06s',
#       'project1'
#   );
# """)

In [3]:
datamart = pd.read_sql_query("SELECT * FROM datamart", con)

In [4]:
datamart['first_commit_ts'] = pd.to_datetime(datamart['first_commit_ts'])
datamart['first_view_ts'] = pd.to_datetime(datamart['first_view_ts'])

In [5]:
datamart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              140 non-null    object        
 1   labname          140 non-null    object        
 2   first_commit_ts  140 non-null    datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 4.5+ KB


In [6]:
test = pd.read_sql_query("""SELECT * 
    FROM datamart
    WHERE first_view_ts IS NOT NULL
    """, con)

In [7]:
test['first_view_ts'] = pd.to_datetime(test['first_view_ts'])
test['first_commit_ts'] = pd.to_datetime(test['first_commit_ts'])

In [8]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              59 non-null     object        
 1   labname          59 non-null     object        
 2   first_commit_ts  59 non-null     datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 2.0+ KB


In [9]:
control = pd.read_sql_query("""SELECT * 
    FROM datamart
    WHERE first_view_ts IS NULL
    """, con)

In [10]:
control.head()

,uid,labname,first_commit_ts,first_view_ts
0,user_4,project1,2020-04-17 05:19:02.744528,None
1,user_4,laba04,2020-04-17 11:33:17.366400,None
2,user_4,laba04s,2020-04-17 11:48:41.992466,None
3,user_2,laba04,2020-04-18 13:42:35.482008,None
4,user_2,laba04s,2020-04-18 13:51:22.291271,None


In [11]:
avg_view = test['first_view_ts'].mean()

In [12]:
control['first_view_ts'] = pd.to_datetime(control['first_view_ts'])
control['first_commit_ts'] = pd.to_datetime(control['first_commit_ts'])

In [13]:
control['first_view_ts']= control['first_view_ts'].fillna(avg_view)

In [14]:
control.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              81 non-null     object        
 1   labname          81 non-null     object        
 2   first_commit_ts  81 non-null     datetime64[ns]
 3   first_view_ts    81 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 2.7+ KB


In [15]:
control.head()

,uid,labname,first_commit_ts,first_view_ts
0,user_4,project1,2020-04-17 05:19:02.744528,2020-04-27 00:40:05.761783552
1,user_4,laba04,2020-04-17 11:33:17.366400,2020-04-27 00:40:05.761783552
2,user_4,laba04s,2020-04-17 11:48:41.992466,2020-04-27 00:40:05.761783552
3,user_2,laba04,2020-04-18 13:42:35.482008,2020-04-27 00:40:05.761783552
4,user_2,laba04s,2020-04-18 13:51:22.291271,2020-04-27 00:40:05.761783552


In [16]:
test.to_sql('test', con, if_exists='replace', index=False)
control.to_sql('control', con, if_exists='replace', index=False)

81

In [17]:
con.close()